# Modelo

In [28]:
import numpy as np
import pandas as pd
import joblib
import os
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED, N_MELS

Elegimos dataset

In [2]:
dataset = 'ciclos_hl256_peak_t_aug_padmin_wd'

## Espectrogramas

Se suelen usar Mel espectrogramas

In [3]:
train_data = np.load(f'./dataset/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((11915, 47616), (11915,), (2850, 47616), (2850,))

### Random Forest

#### Entrenamiento

In [14]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': randint(5, 15),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(10, 20)
}

In [16]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.683 total time= 1.1min
[CV 2/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.675 total time= 1.2min
[CV 3/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.664 total time=  59.2s
[CV 4/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.684 total time=  42.6s
[CV 5/5] END max_depth=11, min_samples_leaf=13, min_samples_split=27;, score=0.673 total time=  58.8s
[CV 1/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.684 total time= 1.0min
[CV 2/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.679 total time=  54.0s
[CV 3/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.661 total time=  49.2s
[CV 4/5] END max_depth=12, min_samples_leaf=14, min_samples_split=21;, score=0.677 total time=  33.1s
[CV 5/5] END max_dept

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....002770552F950>, 'min_samples_leaf': <scipy.stats....00277055A32F0>, 'min_samples_split': <scipy.stats....002770552ED70>}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [17]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,14,12,21,0.676935
0,11,13,27,0.675707
8,14,15,27,0.675439
1,12,14,21,0.675148
3,12,14,18,0.675148
4,12,17,17,0.674225
6,12,15,16,0.672746
9,13,10,25,0.672598
5,10,14,16,0.670882
7,9,10,26,0.666472


In [18]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 14, 'min_samples_leaf': 12, 'min_samples_split': 21}
Best CV score: 0.6769346886796356


In [19]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.93      0.94      6046
           1       0.93      0.94      0.94      5869

    accuracy                           0.94     11915
   macro avg       0.94      0.94      0.94     11915
weighted avg       0.94      0.94      0.94     11915



#### Evaluación

In [20]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.58      0.64      0.61      1452
           1       0.59      0.52      0.55      1398

    accuracy                           0.58      2850
   macro avg       0.59      0.58      0.58      2850
weighted avg       0.59      0.58      0.58      2850



#### Guardado

In [21]:
os.makedirs(f'./modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos/{dataset}/melspec_rf.pkl')

['./modelos/ciclos_hl256_peak_t_aug_padmin_wd/melspec_rf.pkl']

### Gradient Boost

In [28]:
gbc = HistGradientBoostingClassifier(
    max_iter=100,
    random_state=SEED,
    early_stopping=True,
    n_iter_no_change=10,
    learning_rate=0.2,
    l2_regularization=0.5
    )

In [29]:
gbc.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.2
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.5
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [30]:
predict_train = gbc.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.87      0.88      0.88      6046
           1       0.88      0.87      0.87      5869

    accuracy                           0.88     11915
   macro avg       0.88      0.88      0.88     11915
weighted avg       0.88      0.88      0.88     11915



In [31]:
predict_test = gbc.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.60      0.65      0.62      1452
           1       0.60      0.54      0.57      1398

    accuracy                           0.60      2850
   macro avg       0.60      0.60      0.60      2850
weighted avg       0.60      0.60      0.60      2850



In [34]:
joblib.dump(gbc, f'./modelos/{dataset}/melspec_gbc.pkl')

['./modelos/ciclos_hl256_peak_t_aug_padmin_wd/melspec_gbc.pkl']

## Features de Audio

In [35]:
train_data = np.load(f'./dataset/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [36]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((11915, 46), (11915,), (2850, 46), (2850,))

### Random Forest

#### Entrenamiento

In [37]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(13, 20),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(5, 15),
    'max_features': ['sqrt', 'log2', None]
}

In [38]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.713 total time=   1.4s
[CV 2/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.698 total time=   1.2s
[CV 3/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.705 total time=   1.2s
[CV 4/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.713 total time=   1.2s
[CV 5/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.714 total time=   1.3s
[CV 1/5] END max_depth=17, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.723 total time=   8.5s
[CV 2/5] END max_depth=17, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.699 total time=   8.0s
[CV 3/5] END max_depth=17, max_features=None, min_samples_leaf=14, min_samples_split=17;, s

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0027915CB7130>, 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': <scipy.stats....002790E166D50>, 'min_samples_split': <scipy.stats....002790E167550>}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [40]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'param_max_features', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,param_max_features,mean_test_score
26,16,7,15,None,0.716945
9,19,7,19,sqrt,0.716469
41,19,7,20,None,0.716033
3,16,10,19,None,0.715332
43,18,9,20,sqrt,0.714909
24,19,5,23,sqrt,0.714752
12,16,6,24,log2,0.714675
27,15,5,25,None,0.714601
11,19,8,23,log2,0.714466
32,15,7,15,log2,0.714249


In [41]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 16, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 15}
Best CV score: 0.7169454611069874


In [42]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6046
           1       0.93      0.95      0.94      5869

    accuracy                           0.94     11915
   macro avg       0.94      0.94      0.94     11915
weighted avg       0.94      0.94      0.94     11915



#### Evaluación

In [43]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.59      0.86      0.70      1452
           1       0.72      0.36      0.48      1398

    accuracy                           0.62      2850
   macro avg       0.65      0.61      0.59      2850
weighted avg       0.65      0.62      0.59      2850



#### Guardado

In [44]:
joblib.dump(best_model, f'./modelos/{dataset}/features_rf.pkl')

['./modelos/ciclos_hl256_peak_t_aug_padmin_wd/features_rf.pkl']

### Gradient Boosting

In [45]:
gbc = HistGradientBoostingClassifier(
    max_iter=100,
    random_state=SEED,
    early_stopping=True,
    n_iter_no_change=10
    )

In [55]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'l2_regularization': [0, 0.1, 0.3, 0.5, 0.7, 0.9]
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV 1/5] END l2_regularization=0, learning_rate=0.05;, score=0.709 total time=   0.5s
[CV 2/5] END l2_regularization=0, learning_rate=0.05;, score=0.690 total time=   0.4s
[CV 3/5] END l2_regularization=0, learning_rate=0.05;, score=0.697 total time=   0.5s
[CV 4/5] END l2_regularization=0, learning_rate=0.05;, score=0.707 total time=   0.5s
[CV 5/5] END l2_regularization=0, learning_rate=0.05;, score=0.705 total time=   0.6s
[CV 1/5] END l2_regularization=0, learning_rate=0.1;, score=0.713 total time=   0.5s
[CV 2/5] END l2_regularization=0, learning_rate=0.1;, score=0.695 total time=   0.4s
[CV 3/5] END l2_regularization=0, learning_rate=0.1;, score=0.705 total time=   0.4s
[CV 4/5] END l2_regularization=0, learning_rate=0.1;, score=0.701 total time=   0.4s
[CV 5/5] END l2_regularization=0, learning_rate=0.1;, score=0.703 total time=   0.5s
[CV 1/5] END l2_regularization=0, learning_rate=0.2;, score=0.708 total time=   0.3s

,estimator,HistGradientB...ndom_state=42)
,param_grid,"{'l2_regularization': [0, 0.1, ...], 'learning_rate': [0.05, 0.1, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [56]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate', 'param_l2_regularization', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_learning_rate,param_l2_regularization,mean_test_score
13,0.10,0.7,0.710430
4,0.10,0.1,0.706303
7,0.10,0.3,0.705903
10,0.10,0.5,0.705798
11,0.20,0.5,0.705068
16,0.10,0.9,0.704179
1,0.10,0.0,0.703394
0,0.05,0.0,0.701807
5,0.20,0.1,0.701605
6,0.05,0.3,0.701191


In [57]:
best_model = rnd_search.best_estimator_

In [59]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.85      0.83      0.84      6046
           1       0.83      0.85      0.84      5869

    accuracy                           0.84     11915
   macro avg       0.84      0.84      0.84     11915
weighted avg       0.84      0.84      0.84     11915



In [60]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.58      0.85      0.69      1452
           1       0.70      0.37      0.48      1398

    accuracy                           0.61      2850
   macro avg       0.64      0.61      0.59      2850
weighted avg       0.64      0.61      0.59      2850



In [61]:
joblib.dump(best_model, f'./modelos/{dataset}/features_gbc.pkl')

['./modelos/ciclos_hl256_peak_t_aug_padmin_wd/features_gbc.pkl']

## Imagenes de Mel Espectrogramas

In [ ]:
train_dir = f'./dataset/{dataset}/mel_images/train'
test_dir = f'./dataset/{dataset}/mel_images/test'

In [30]:
def load_image_dataset_pil(img_dir, to_gray=True, flatten=False, target_size=None):
    df = pd.read_csv(os.path.join(img_dir, 'labels.csv'))
    X, y = [], []
    for _, row in df.iterrows():
        path = os.path.join(img_dir, row['filename'])
        img = Image.open(path)
        
        if to_gray:
            img = img.convert('L')  # convierte a escala de grises (1 canal)
        
        if target_size:
            img = img.resize(target_size, Image.BICUBIC)
        
        img = np.array(img, dtype=np.float32)
        
        if flatten:
            img = img.flatten()
        
        X.append(img)
        y.append(row['label'])
    
    return np.array(X), np.array(y)

In [31]:
X_train, y_train = load_image_dataset_pil(train_dir, to_gray=True, flatten=True, target_size=(256, N_MELS))
X_test, y_test = load_image_dataset_pil(test_dir, to_gray=True, flatten=True, target_size=(256, N_MELS))

### Random Forest

In [48]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')

param_distributions = {
    'max_depth': randint(5, 15),
    'min_samples_split': randint(10, 20),
    'min_samples_leaf': randint(5, 10)
}

In [49]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=11, min_samples_leaf=8, min_samples_split=17;, score=0.680 total time=  18.0s
[CV 2/5] END max_depth=11, min_samples_leaf=8, min_samples_split=17;, score=0.674 total time=  23.7s
[CV 3/5] END max_depth=11, min_samples_leaf=8, min_samples_split=17;, score=0.659 total time=  39.8s
[CV 4/5] END max_depth=11, min_samples_leaf=8, min_samples_split=17;, score=0.666 total time=  39.0s
[CV 5/5] END max_depth=11, min_samples_leaf=8, min_samples_split=17;, score=0.676 total time=  38.7s
[CV 1/5] END max_depth=9, min_samples_leaf=6, min_samples_split=12;, score=0.674 total time=  34.3s
[CV 2/5] END max_depth=9, min_samples_leaf=6, min_samples_split=12;, score=0.666 total time=  33.1s
[CV 3/5] END max_depth=9, min_samples_leaf=6, min_samples_split=12;, score=0.654 total time=  34.5s
[CV 4/5] END max_depth=9, min_samples_leaf=6, min_samples_split=12;, score=0.659 total time=  15.1s
[CV 5/5] END max_depth=9, min_samp

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....001F6952207C0>, 'min_samples_leaf': <scipy.stats....001F6950B7550>, 'min_samples_split': <scipy.stats....001F6952209E0>}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [50]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
4,12,7,15,0.672478
0,11,8,17,0.671235
2,11,7,17,0.667070
6,10,6,14,0.664937
1,9,6,12,0.664492
8,10,9,18,0.664443
3,9,8,17,0.664030
5,9,6,17,0.663687
9,5,7,19,0.643356
7,5,8,19,0.643218


In [59]:
best_model = rnd_search.best_estimator_

In [60]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.98      0.93      0.95      6046
           1       0.93      0.98      0.96      5869

    accuracy                           0.96     11915
   macro avg       0.96      0.96      0.96     11915
weighted avg       0.96      0.96      0.96     11915



In [61]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.70      0.65      0.68       663
           1       0.66      0.71      0.69       633

    accuracy                           0.68      1296
   macro avg       0.68      0.68      0.68      1296
weighted avg       0.68      0.68      0.68      1296



In [62]:
joblib.dump(best_model, f'./modelos/{dataset}/melspec_img_rf.pkl')

['./modelos/ciclos_hl256_peak_t_aug_padmin_wd/melspec_img_rf.pkl']